# PHASE 4 — UNSUPERVISED LEARNING


# Day 21 — K-Means Clustering


## 1. Learning Objectives
By the end of this notebook, you will be able to:
- Explain the difference between Supervised and Unsupervised learning.
- Understand how K-Means finds hidden groups without any labels.
- Explain Centroids and Inertia.
- Use the **Elbow Method** to figure out how many clusters exist in unknown data.


## 2. Prerequisites
- Day 17 (K-Nearest Neighbors / Distance Math).
- Day 4 (StandardScaler).


## 3. Concept: Unsupervised Learning
Up until today, every dataset had a target `y` (Price, Fraud, Spam). We trained models to predict `y` from `X`. This is called **Supervised Learning**.

What if there is no `y`? What if I just hand you 10,000 customer records (Age, Salary, Spending Score) and say: *"Find some interesting patterns in here."*
This is **Unsupervised Learning**. You are exploring the data blindly to discover hidden structures.


## 4. Concept: K-Means Clustering
**Clustering** is the task of grouping similar data points together. The most famous algorithm is **K-Means**.

How it works:
1. You tell the algorithm how many clusters you want (e.g., $K=3$).
2. It drops 3 random pins (called **Centroids**) onto the dataset.
3. Every data point looks to see which Centroid it is closest to and joins that Centroid's team.
4. The Centroid then moves to the exact mathematical center of all the points on its team.
5. Repeat steps 3 and 4 until the Centroids stop moving. 

*Because K-Means uses physical distance to determine which team a point belongs to, you MUST scale your data!*


## 5. Scikit-learn API
```python
from sklearn.cluster import KMeans
model = KMeans(n_clusters=3)
model.fit(X) # Notice there is no 'y' !!
```


## 6. Simple Example
Let's generate a dataset that clearly has 4 blobs of data, but we won't tell the algorithm that. We will ask it to find 4 clusters.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

# 1. Generate Data (We throw away the 'y' labels!)
X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.6, random_state=0)

# 2. Train K-Means (Notice we ONLY pass X)
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(X)

# 3. Get the Labels (The team assignments)
team_labels = kmeans.labels_

# 4. Get the Centroid coordinates
centroids = kmeans.cluster_centers_

print('First 10 team assignments:', team_labels[:10])


## 7. Code Walkthrough
- We called `kmeans.fit(X)`. No `y` was provided.
- The algorithm assigned every row an integer (0, 1, 2, or 3) indicating which cluster it belongs to. These are accessed via `.labels_`.
- It also stored the final resting place of the 4 pins inside `.cluster_centers_`.


## 8. Experiment
Let's plot the data and color it based on the teams K-Means discovered. We'll also draw red X's where the final Centroids landed.


In [ ]:
plt.figure(figsize=(8, 5))

# Plot the data points, colored by their K-Means label
plt.scatter(X[:, 0], X[:, 1], c=team_labels, cmap='viridis', s=50, alpha=0.6)

# Plot the Centroids
plt.scatter(centroids[:, 0], centroids[:, 1], c='red', s=200, marker='X', label='Centroids')

plt.title('K-Means Clustering (K=4)')
plt.legend()
plt.show()


> It perfectly identified the 4 distinct blobs and placed a Centroid exactly in the middle of each one!


## 9. Prediction Exercise
Read the following code, but **DO NOT RUN IT YET**.


In [ ]:
kmeans_k2 = KMeans(n_clusters=2, random_state=42)
kmeans_k2.fit(X)


> **Question:** We know there are 4 distinct blobs. What happens if we force K-Means to find only 2 clusters?

**Think before running the next cell!**


In [ ]:
plt.figure(figsize=(5, 3))
plt.scatter(X[:, 0], X[:, 1], c=kmeans_k2.labels_, cmap='viridis', alpha=0.6)
plt.scatter(kmeans_k2.cluster_centers_[:, 0], kmeans_k2.cluster_centers_[:, 1], c='red', s=200, marker='X')
plt.title('Forced to find K=2')
plt.show()
print('It grouped the two left blobs into one massive cluster, and the two right blobs into another.')
print('K-Means will ALWAYS find exactly the number of clusters you ask it to, even if that number is wrong.')


## 10. The Elbow Method (Finding the true K)
In the real world, you don't know how many clusters exist. How do you find the right $K$?

We look at **Inertia** (`kmeans.inertia_`). Inertia measures how tightly packed the clusters are. 
If Inertia is 0, every data point is sitting directly on top of a Centroid. 
As $K$ increases, Inertia always decreases. But we don't want $K=300$. 

We train the model for $K=1, 2, 3... 10$ and plot the Inertia. The graph will look like an arm. The "Elbow" of the arm represents the optimal $K$ — the point where adding more clusters stops providing massive improvements.


## 11. Coding Exercise
Write a `for` loop that trains a KMeans model for every $K$ from 1 to 10. Append the `.inertia_` of each model to a list. Finally, plot the list. Where is the elbow?


In [ ]:
# YOUR CODE HERE
inertias = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, marker='o', linestyle='--')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Tightness)')
plt.title('The Elbow Method')
plt.xticks(K_range)
plt.grid(True)
plt.show()

print('Notice how the massive drops stop at K=4. That is the elbow! This mathematically proves there are 4 clusters.')


## 12. Debugging Challenge
A data analyst is trying to group customers based on Age (18-90) and Salary ($20,000 - $200,000). The model is completely ignoring the Age variable and only grouping people based on Salary. Why?


In [ ]:
# Conceptual Bug
print('Error: The model is ignoring Age entirely.')
print('Why? K-Means uses physical Euclidean distance.')
print('A difference of 50 years in Age is mathematically crushed by a difference of $50,000 in Salary.')


> **Rule:** You MUST use `StandardScaler` or `MinMaxScaler` before running K-Means so that all features contribute equally to the distance calculation.


## 13. Model Evaluation
Evaluating unsupervised models is notoriously difficult because there is no "ground truth" to compare against. 
- **Inertia**: Lower is better, but it's heavily influenced by the number of clusters.
- **Silhouette Score**: We will learn this advanced metric tomorrow.


## 14. Real-World Example
**Customer Segmentation**: A marketing team has a database of 1 million users. They don't know anything about them. They run K-Means (K=5) on their purchasing habits. 
The algorithm blindly creates 5 clusters. The analysts look at the clusters and realize:
- Cluster 0: "Bargain Hunters" (only buy on sale)
- Cluster 1: "Whales" (spend massive amounts)
- Cluster 2: "Window Shoppers" (browse a lot, buy nothing)
The marketing team then creates 3 totally different email campaigns tailored specifically to those exact personas!


## 15. Mini Project
Build a Pipeline with `StandardScaler` and `KMeans(n_clusters=3)`. Generate a random dataset with 3 features and heavily distorted scales. Fit the pipeline and print the cluster centers (Centroids) for the scaled data.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X_distorted = np.random.rand(150, 3)
X_distorted[:, 0] *= 10000 # Feature 0 is massive

kmeans_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('kmeans', KMeans(n_clusters=3, random_state=42))
])

kmeans_pipe.fit(X_distorted)

# To get the centers, you have to extract the trained kmeans model from the pipeline
trained_kmeans = kmeans_pipe.named_steps['kmeans']
print('Scaled Centroid Coordinates:\n', trained_kmeans.cluster_centers_)


## 16. Common Mistakes
- **Not Scaling Data**: The #1 mistake. K-Means will fail spectacularly on unscaled data.
- **Choosing K arbitrarily**: Always use the Elbow Method to justify your choice of K.
- **Interpreting Clusters**: K-Means doesn't know *what* it found. It just groups numbers. It is up to human intuition to look at the groups and give them meaning (e.g., "Ah, these are the Bargain Hunters").


## 17. Interview Questions
- **Beginner**: What is the difference between Supervised and Unsupervised Learning? (Answer: Supervised has labels/targets, Unsupervised explores raw data without labels).
- **Intermediate**: Explain the Elbow Method. (Answer: Plot Inertia for various Ks. Find the "elbow" where adding more clusters yields diminishing returns in tightness).
- **Advanced**: Why does K-Means scale poorly to massive datasets? (Answer: Because in every single iteration, it has to calculate the physical distance between *every* point and *every* centroid. For millions of rows, this becomes extremely slow).


## 18. Knowledge Check
- What property stores the assignments of every data point? (`.labels_`)
- What metric measures how tightly packed the clusters are? (Inertia)


## 19. Summary
- **Unsupervised Learning** finds hidden structures without labels.
- **K-Means** finds clusters by moving Centroids to the middle of data groupings.
- **Inertia** measures tightness.
- **The Elbow Method** is the standard way to find the optimal $K$.
- **Scaling** is absolutely mandatory.


## 20. Homework
Load the `load_iris` dataset, but throw away the `target` array entirely! Build a pipeline with `StandardScaler` and `KMeans`. Run the Elbow method on the iris data. Does the elbow appear at $K=3$? (Spoiler: It should, since there are 3 species of Iris flowers!).
